# 04 — Dengue Monthly Conversion

## Dengue Disease Prediction for Sri Lanka

### Purpose

The raw dengue dataset is reported weekly, while the forecasting framework uses monthly observations. This notebook converts weekly dengue case counts into monthly dengue case counts while preserving the temporal information contained in reporting-period dates.

Two monthly conversion strategies are examined:

1. **End-date assignment** — assigns the complete weekly case count to the month containing the reporting-period end date.
2. **Day-weighted proportional allocation** — distributes weekly cases across calendar months according to the number of reporting days falling within each month.

The proportional method is used to reduce temporal boundary distortion when a weekly reporting period crosses a month boundary.

No raw data files are modified.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

print("Project root:", PROJECT_ROOT)
print("Interim directory:", INTERIM_DIR)

Project root: d:\Dengue_Project
Interim directory: d:\Dengue_Project\data\interim


In [3]:
dengue_file = (
    INTERIM_DIR /
    "dengue_weekly_geography_standardized.csv"
)
print("File exists:", dengue_file.exists())

File exists: True


In [4]:
dengue = pd.read_csv(dengue_file)
print("Shape:", dengue.shape)
dengue.head()

Shape: (20306, 9)


,year,week,start.date,end.date,district,cases,source_area,reporting_area,administrative_district
0,2010,1,1/2/2010,1/8/2010,Colombo,176,Colombo,Colombo,Colombo
1,2010,1,1/2/2010,1/8/2010,Gampaha,153,Gampaha,Gampaha,Gampaha
2,2010,1,1/2/2010,1/8/2010,Kalutara,32,Kalutara,Kalutara,Kalutara
3,2010,1,1/2/2010,1/8/2010,Kandy,68,Kandy,Kandy,Kandy
4,2010,1,1/2/2010,1/8/2010,Matale,25,Matale,Matale,Matale


In [5]:
required_columns = [
    "year",
    "week",
    "start.date",
    "end.date",
    "cases",
    "reporting_area",
    "administrative_district"
]
missing_columns = [
    column
    for column in required_columns
    if column not in dengue.columns
]

print("Missing required columns:", missing_columns)

Missing required columns: []


In [6]:
dengue[
    ["start.date", "end.date"]
].head(10)

,start.date,end.date
0,1/2/2010,1/8/2010
1,1/2/2010,1/8/2010
2,1/2/2010,1/8/2010
3,1/2/2010,1/8/2010
4,1/2/2010,1/8/2010
5,1/2/2010,1/8/2010
6,1/2/2010,1/8/2010
7,1/2/2010,1/8/2010
8,1/2/2010,1/8/2010
9,1/2/2010,1/8/2010


In [7]:
print("Start date dtype:", dengue["start.date"].dtype)
print("End-date dtype:", dengue["end.date"].dtype)

Start date dtype: str
End-date dtype: str


In [8]:
dengue["start_date"] = pd.to_datetime(
    dengue["start.date"],
    errors="coerce"
)
dengue["end_date"] = pd.to_datetime(
    dengue["end.date"],
    errors="coerce"
)

In [9]:
print(
    "Invalid start dates:",
    dengue["start_date"].isna().sum()
)
print(
    "Invalid end dates:",
    dengue["end_date"].isna().sum()
)

Invalid start dates: 0
Invalid end dates: 0


Confirm seven-day reporting periods again

In [10]:
dengue["reporting_days"] = (
    dengue["end_date"] -
    dengue["start_date"]
).dt.days + 1
dengue["reporting_days"].value_counts().sort_index()

reporting_days
7    20306
Name: count, dtype: int64

Determine how many weeks cross month boundaries

In [11]:
dengue["crosses_month"] = (
    dengue["start_date"].dt.to_period("M") !=
    dengue["end_date"].dt.to_period("M")
)
print(
    "Weekly observations crossing month boundaries:",
    dengue["crosses_month"].sum()
)
print(
    "Percentage crossing month boundaries:",
    round(
        dengue["crosses_month"].mean() * 100,
        2
    ),
    "%"
)

Weekly observations crossing month boundaries: 4030
Percentage crossing month boundaries: 19.85 %


In [12]:
dengue.loc[
    dengue["crosses_month"],
    [
        "year",
        "week",
        "reporting_area",
        "start_date",
        "end_date",
        "cases"
    ]
].head(20)

,year,week,reporting_area,start_date,end_date,cases
104,2010,5,Colombo,2010-01-30,2010-02-05,81
105,2010,5,Gampaha,2010-01-30,2010-02-05,89
106,2010,5,Kalutara,2010-01-30,2010-02-05,31
107,2010,5,Kandy,2010-01-30,2010-02-05,31
108,2010,5,Matale,2010-01-30,2010-02-05,44
109,2010,5,Nuwara Eliya,2010-01-30,2010-02-05,2
110,2010,5,Galle,2010-01-30,2010-02-05,15
111,2010,5,Hambantota,2010-01-30,2010-02-05,22
112,2010,5,Matara,2010-01-30,2010-02-05,9
113,2010,5,Jaffna,2010-01-30,2010-02-05,329


Check year-boundary weeks too

In [13]:
dengue["crosses_year"] = (
    dengue["start_date"].dt.year !=
    dengue["end_date"].dt.year
)
print(
    "Weekly observations crossing calendar-year boundaries:",
    dengue["crosses_year"].sum()
)

Weekly observations crossing calendar-year boundaries: 312


In [14]:
dengue.loc[
    dengue["crosses_year"],
    [
        "year",
        "week",
        "reporting_area",
        "start_date",
        "end_date",
        "cases"
    ]
]

,year,week,reporting_area,start_date,end_date,cases
2704,2012,1,Colombo,2011-12-31,2012-01-06,219
2705,2012,1,Gampaha,2011-12-31,2012-01-06,117
2706,2012,1,Kalutara,2011-12-31,2012-01-06,36
2707,2012,1,Kandy,2011-12-31,2012-01-06,60
2708,2012,1,Matale,2011-12-31,2012-01-06,9
...,...,...,...,...,...,...
19001,2024,2,Badulla,2023-12-30,2024-01-05,111
19002,2024,2,Monaragala,2023-12-30,2024-01-05,42
19003,2024,2,Ratnapura,2023-12-30,2024-01-05,76
19004,2024,2,Kegalle,2023-12-30,2024-01-05,112


Calculate the original total dengue cases

In [15]:
original_total_cases = dengue["cases"].sum()
print(
    "Original weekly dengue case total:",
    original_total_cases
)

Original weekly dengue case total: 714747


## 1. Baseline Monthly Conversion: End-Date Assignment

As a simple baseline, each weekly dengue observation is assigned completely to the calendar month containing its reporting period end date.

This approach is easy to implement but can introduce temporal boundary distortion because a reporting week may contain days from two different months.

In [16]:
dengue_enddate = dengue.copy()
dengue_enddate["month"] = (
    dengue_enddate["end_date"]
    .dt.to_period("M")
)

In [17]:
dengue_enddate[
    [
        "start_date",
        "end_date",
        "month",
        "reporting_area",
        "cases"
    ]
].head()

,start_date,end_date,month,reporting_area,cases
0,2010-01-02,2010-01-08,2010-01,Colombo,176
1,2010-01-02,2010-01-08,2010-01,Gampaha,153
2,2010-01-02,2010-01-08,2010-01,Kalutara,32
3,2010-01-02,2010-01-08,2010-01,Kandy,68
4,2010-01-02,2010-01-08,2010-01,Matale,25


Aggregate the baseline monthly cases

In [18]:
monthly_enddate = (
    dengue_enddate
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )["cases"]
    .sum()
)
monthly_enddate = monthly_enddate.rename(
    columns={"cases": "dengue_cases_enddate"}
)

monthly_enddate.head()

,reporting_area,month,dengue_cases_enddate
0,Ampara,2010-01,3
1,Ampara,2010-02,8
2,Ampara,2010-03,8
3,Ampara,2010-04,3
4,Ampara,2010-05,6


In [19]:
print("Monthly baseline rows:", len(monthly_enddate))
print(
    "Reporting areas:",
    monthly_enddate["reporting_area"].nunique()
)
print(
    "Months:",
    monthly_enddate["month"].nunique()
)

Monthly baseline rows: 4680
Reporting areas: 26
Months: 180


Check baseline case conservation

In [20]:
baseline_total_cases = (
    monthly_enddate["dengue_cases_enddate"].sum()
)
print("Original total:", original_total_cases)
print("Baseline monthly total:", baseline_total_cases)
print(
    "Difference:",
    baseline_total_cases - original_total_cases
)

Original total: 714747
Baseline monthly total: 714747
Difference: 0


## 2. Day Weighted Proportional Monthly Allocation

For weekly reporting periods contained entirely within one calendar month, the complete weekly case count is assigned to that month.

For reporting periods crossing a month boundary, the weekly case count is distributed proportionally according to the number of reporting days occurring in each calendar month.

For a weekly case count \(C\), the cases allocated to month \(m\) are:

\[
C_m = C \times \frac{d_m}{D}
\]

where:

- \(C_m\) = cases allocated to month \(m\)
- \(C\) = total cases reported for the week
- \(d_m\) = number of reporting days falling within month \(m\)
- \(D\) = total number of days in the reporting period

For the present dataset, \(D = 7\).

This method assumes that cases are uniformly distributed across the reporting days when daily case counts are unavailable.

Create daily allocation records

In [21]:
daily_records = []

for _, row in dengue.iterrows():

    dates = pd.date_range(
        start=row["start_date"],
        end=row["end_date"],
        freq="D"
    )

    daily_case_share = (
        row["cases"] / len(dates)
    )

    for date in dates:

        daily_records.append({
            "date": date,
            "reporting_area": row["reporting_area"],
            "administrative_district": row["administrative_district"],
            "weekly_cases": row["cases"],
            "allocated_cases": daily_case_share
        })

In [22]:
daily_allocated = pd.DataFrame(daily_records)

print("Daily allocation rows:", len(daily_allocated))

daily_allocated.head(10)

Daily allocation rows: 142142


,date,reporting_area,administrative_district,weekly_cases,allocated_cases
0,2010-01-02,Colombo,Colombo,176,25.142857
1,2010-01-03,Colombo,Colombo,176,25.142857
2,2010-01-04,Colombo,Colombo,176,25.142857
3,2010-01-05,Colombo,Colombo,176,25.142857
4,2010-01-06,Colombo,Colombo,176,25.142857
5,2010-01-07,Colombo,Colombo,176,25.142857
6,2010-01-08,Colombo,Colombo,176,25.142857
7,2010-01-02,Gampaha,Gampaha,153,21.857143
8,2010-01-03,Gampaha,Gampaha,153,21.857143
9,2010-01-04,Gampaha,Gampaha,153,21.857143


Validate the daily expansion

In [23]:
expected_daily_rows = len(dengue) * 7
print("Expected daily rows:", expected_daily_rows)
print("Actual daily rows:", len(daily_allocated))
print(
    "Daily row count correct:",
    expected_daily_rows == len(daily_allocated)
)

Expected daily rows: 142142
Actual daily rows: 142142
Daily row count correct: True


In [24]:
daily_allocated_total = (
    daily_allocated["allocated_cases"].sum()
)
print(
    "Original weekly total:",
    original_total_cases
)
print(
    "Daily allocated total:",
    daily_allocated_total
)
print(
    "Difference:",
    daily_allocated_total -
    original_total_cases
)

Original weekly total: 714747
Daily allocated total: 714747.0
Difference: 0.0


In [25]:
print(
    "Cases preserved:",
    np.isclose(
        daily_allocated_total,
        original_total_cases
    )
)

Cases preserved: True


Create calendar month from actual dates

In [26]:
daily_allocated["month"] = (
    daily_allocated["date"]
    .dt.to_period("M")
)

In [27]:
daily_allocated[
    [
        "date",
        "month",
        "reporting_area",
        "allocated_cases"
    ]
].head()

,date,month,reporting_area,allocated_cases
0,2010-01-02,2010-01,Colombo,25.142857
1,2010-01-03,2010-01,Colombo,25.142857
2,2010-01-04,2010-01,Colombo,25.142857
3,2010-01-05,2010-01,Colombo,25.142857
4,2010-01-06,2010-01,Colombo,25.142857


Aggregate daily contributions into monthly cases

In [28]:
monthly_proportional = (
    daily_allocated
    .groupby(
        ["reporting_area", "month"],
        as_index=False
    )["allocated_cases"]
    .sum()
)

monthly_proportional = monthly_proportional.rename(
    columns={
        "allocated_cases":
        "dengue_cases_proportional"
    }
)

monthly_proportional.head()

,reporting_area,month,dengue_cases_proportional
0,Ampara,2010-01,3.857143
1,Ampara,2010-02,7.142857
2,Ampara,2010-03,8.714286
3,Ampara,2010-04,2.285714
4,Ampara,2010-05,6.000000


Validate total case conservation again

In [29]:
proportional_total_cases = (
    monthly_proportional[
        "dengue_cases_proportional"
    ].sum()
)
print(
    "Original weekly total:",
    original_total_cases
)
print(
    "Proportional monthly total:",
    proportional_total_cases
)
print(
    "Cases preserved:",
    np.isclose(
        original_total_cases,
        proportional_total_cases
    )
)

Original weekly total: 714747
Proportional monthly total: 714747.0
Cases preserved: True


Validate monthly coverage

In [30]:
print(
    "Reporting areas:",
    monthly_proportional[
        "reporting_area"
    ].nunique()
)
print(
    "Unique months:",
    monthly_proportional[
        "month"
    ].nunique()
)
print(
    "First month:",
    monthly_proportional["month"].min()
)
print(
    "Last month:",
    monthly_proportional["month"].max()
)

print(
    "Total monthly rows:",
    len(monthly_proportional)
)

Reporting areas: 26
Unique months: 180
First month: 2010-01
Last month: 2024-12
Total monthly rows: 4680


In [31]:
months_per_area = (
    monthly_proportional
    .groupby("reporting_area")["month"]
    .nunique()
    .sort_values()
)
months_per_area

reporting_area
Ampara          180
Anuradhapura    180
Badulla         180
Batticaloa      180
Colombo         180
Galle           180
Gampaha         180
Hambantota      180
Jaffna          180
Kalmunai        180
Kalutara        180
Kandy           180
Kegalle         180
Kilinochchi     180
Kurunegala      180
Mannar          180
Matale          180
Matara          180
Monaragala      180
Mullaitivu      180
Nuwara Eliya    180
Polonnaruwa     180
Puttalam        180
Ratnapura       180
Trincomalee     180
Vavuniya        180
Name: month, dtype: int64

In [32]:
print(
    "Minimum months per area:",
    months_per_area.min()
)
print(
    "Maximum months per area:",
    months_per_area.max()
)

Minimum months per area: 180
Maximum months per area: 180


Compare baseline vs proportional conversion

In [33]:
monthly_comparison = monthly_enddate.merge(
    monthly_proportional,
    on=["reporting_area", "month"],
    how="outer"
)

In [34]:
monthly_comparison["difference"] = (
    monthly_comparison[
        "dengue_cases_proportional"
    ] -
    monthly_comparison[
        "dengue_cases_enddate"
    ]
)
monthly_comparison.head()

,reporting_area,month,dengue_cases_enddate,dengue_cases_proportional,difference
0,Ampara,2010-01,3,3.857143,0.857143
1,Ampara,2010-02,8,7.142857,-0.857143
2,Ampara,2010-03,8,8.714286,0.714286
3,Ampara,2010-04,3,2.285714,-0.714286
4,Ampara,2010-05,6,6.000000,0.000000


In [35]:
different_months = monthly_comparison[
    ~np.isclose(
        monthly_comparison[
            "dengue_cases_enddate"
        ].fillna(0),
        monthly_comparison[
            "dengue_cases_proportional"
        ].fillna(0)
    )
]
print(
    "Monthly area observations affected by conversion method:",
    len(different_months)
)

Monthly area observations affected by conversion method: 4269


Inspect the largest differences

In [36]:
monthly_comparison["absolute_difference"] = (
    monthly_comparison["difference"].abs()
)
monthly_comparison.sort_values(
    "absolute_difference",
    ascending=False
).head(20)

,reporting_area,month,dengue_cases_enddate,dengue_cases_proportional,difference,absolute_difference
810,Colombo,2017-07,7471,8205.142857,734.142857,734.142857
809,Colombo,2017-06,5372,4785.571429,-586.428571,586.428571
838,Colombo,2019-11,4217,3633.000000,-584.000000,584.000000
1170,Gampaha,2017-07,9039,9602.571429,563.571429,563.571429
1169,Gampaha,2017-06,4901,4380.285714,-520.714286,520.714286
811,Colombo,2017-08,3620,3192.714286,-427.285714,427.285714
837,Colombo,2019-10,2593,3014.285714,421.285714,421.285714
881,Colombo,2023-06,2252,1862.000000,-390.000000,390.000000
1198,Gampaha,2019-11,2495,2117.714286,-377.285714,377.285714
738,Colombo,2011-07,1555,1185.000000,-370.000000,370.000000


## 3. Monthly Conversion Validation

The weekly dengue dataset contains 20,306 reporting area observations. Of these, 4,030 observations (19.85%) cross a calendar month boundary, demonstrating that direct assignment of complete weekly case counts to a single month can alter the temporal distribution of dengue cases.

Both the end date baseline method and the day weighted proportional method preserve the overall dengue case total of 714,747 cases.

The proportional conversion produces complete monthly coverage from January 2010 through December 2024 for all 26 reporting areas, resulting in 4,680 reporting area month observations.

Comparison of the two conversion approaches shows that 4,269 of the 4,680 monthly reporting area observations differ between the end date and proportional methods. Therefore, the method used to convert weekly dengue observations into monthly observations has a substantial effect on the resulting monthly time series.

The proportional method is retained for subsequent preprocessing because it explicitly accounts for reporting periods that cross calendar month boundaries. This method assumes that cases are uniformly distributed across the seven days of each weekly reporting period.

In [37]:
comparison_stats = pd.Series({
    "Total monthly observations":
        len(monthly_comparison),
    "Affected monthly observations":
        len(different_months),
    "Affected percentage":
        (
            len(different_months) /
            len(monthly_comparison)
        ) * 100,
    "Mean absolute difference":
        monthly_comparison[
            "absolute_difference"
        ].mean(),
    "Maximum absolute difference":
        monthly_comparison[
            "absolute_difference"
        ].max()
})

comparison_stats

Total monthly observations       4680.000000
Affected monthly observations    4269.000000
Affected percentage                91.217949
Mean absolute difference           16.199145
Maximum absolute difference       734.142857
dtype: float64

In [38]:
largest_difference = (
    monthly_comparison
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    .head(1)
)
largest_difference

,reporting_area,month,dengue_cases_enddate,dengue_cases_proportional,difference,absolute_difference
810,Colombo,2017-07,7471,8205.142857,734.142857,734.142857


Calculate percentage difference for affected observations

In [39]:
affected_percentage = (
    len(different_months) /
    len(monthly_comparison)
) * 100
print(
    "Percentage of monthly observations affected:",
    round(affected_percentage, 2),
    "%"
)

Percentage of monthly observations affected: 91.22 %


## 4. Conversion to Administrative District Level

The standardized dengue dataset contains 26 health reporting areas, while the population dataset contains 25 administrative districts.

For integration with district level population data, the proportionally allocated dengue observations are therefore aggregated to the 25 administrative districts.

Dengue case counts are additive. Consequently, where multiple health reporting areas correspond to the same administrative district, their allocated case counts are summed.

In particular, the Ampara and Kalmunai health reporting areas contribute to the Ampara administrative district dengue total.

No comparable aggregation rule is assumed for weather variables in this notebook because different weather variables require variable-specific aggregation strategies.

In [40]:
monthly_admin_proportional = (
    daily_allocated
    .groupby(
        [
            "administrative_district",
            "month"
        ],
        as_index=False
    )["allocated_cases"]
    .sum()
)
monthly_admin_proportional = (
    monthly_admin_proportional.rename(
        columns={
            "allocated_cases":
            "dengue_cases"
        }
    )
)
monthly_admin_proportional.head()

,administrative_district,month,dengue_cases
0,Ampara,2010-01,116.428571
1,Ampara,2010-02,124.428571
2,Ampara,2010-03,72.857143
3,Ampara,2010-04,37.285714
4,Ampara,2010-05,13.714286


Check number of administrative districts

In [41]:
print(
    "Administrative districts:",
    monthly_admin_proportional[
        "administrative_district"
    ].nunique()
)
print(
    "Unique months:",
    monthly_admin_proportional[
        "month"
    ].nunique()
)
print(
    "Total rows:",
    len(monthly_admin_proportional)
)

Administrative districts: 25
Unique months: 180
Total rows: 4500


Check temporal coverage

In [42]:
print(
    "First month:",
    monthly_admin_proportional[
        "month"
    ].min()
)
print(
    "Last month:",
    monthly_admin_proportional[
        "month"
    ].max()
)

First month: 2010-01
Last month: 2024-12


Check months per administrative district

In [43]:
admin_months_per_district = (
    monthly_admin_proportional
    .groupby(
        "administrative_district"
    )["month"]
    .nunique()
    .sort_values()
)
admin_months_per_district

administrative_district
Ampara          180
Anuradhapura    180
Badulla         180
Batticaloa      180
Colombo         180
Galle           180
Gampaha         180
Hambantota      180
Jaffna          180
Kalutara        180
Kandy           180
Kegalle         180
Kilinochchi     180
Kurunegala      180
Mannar          180
Matale          180
Matara          180
Monaragala      180
Mullaitivu      180
Nuwara Eliya    180
Polonnaruwa     180
Puttalam        180
Ratnapura       180
Trincomalee     180
Vavuniya        180
Name: month, dtype: int64

In [44]:
admin_total_cases = (
    monthly_admin_proportional[
        "dengue_cases"
    ].sum()
)
print(
    "Original weekly total:",
    original_total_cases
)
print(
    "26-area proportional total:",
    proportional_total_cases
)
print(
    "25-district proportional total:",
    admin_total_cases
)
print(
    "Cases preserved after district aggregation:",
    np.isclose(
        admin_total_cases,
        original_total_cases
    )
)

Original weekly total: 714747
26-area proportional total: 714747.0
25-district proportional total: 714747.0
Cases preserved after district aggregation: True


Check duplicate district month keys

In [45]:
duplicate_admin_months = (
    monthly_admin_proportional
    .duplicated(
        subset=[
            "administrative_district",
            "month"
        ]
    )
    .sum()
)
print(
    "Duplicate district-month keys:",
    duplicate_admin_months
)

Duplicate district-month keys: 0


Check missing values

In [46]:
print(
    "Missing district values:",
    monthly_admin_proportional[
        "administrative_district"
    ].isna().sum()
)
print(
    "Missing month values:",
    monthly_admin_proportional[
        "month"
    ].isna().sum()
)
print(
    "Missing dengue case values:",
    monthly_admin_proportional[
        "dengue_cases"
    ].isna().sum()
)

Missing district values: 0
Missing month values: 0
Missing dengue case values: 0


Check for negative values

In [47]:
negative_monthly_cases = (
    monthly_admin_proportional[
        "dengue_cases"
    ] < 0
).sum()
print(
    "Negative monthly dengue values:",
    negative_monthly_cases
)

Negative monthly dengue values: 0


Verify Ampara aggregation specifically

In [48]:
ampara_reporting_check = (
    monthly_proportional[
        monthly_proportional[
            "reporting_area"
        ].isin(
            ["Ampara", "Kalmunai"]
        )
    ]
    .pivot(
        index="month",
        columns="reporting_area",
        values="dengue_cases_proportional"
    )
)

ampara_reporting_check.head()

reporting_area,Ampara,Kalmunai
month,,
2010-01,3.857143,112.571429
2010-02,7.142857,117.285714
2010-03,8.714286,64.142857
2010-04,2.285714,35.000000
2010-05,6.000000,7.714286


In [49]:
ampara_reporting_check[
    "expected_admin_ampara"
] = (
    ampara_reporting_check["Ampara"] +
    ampara_reporting_check["Kalmunai"]
)
ampara_reporting_check.head()

reporting_area,Ampara,Kalmunai,expected_admin_ampara
month,,,
2010-01,3.857143,112.571429,116.428571
2010-02,7.142857,117.285714,124.428571
2010-03,8.714286,64.142857,72.857143
2010-04,2.285714,35.000000,37.285714
2010-05,6.000000,7.714286,13.714286


In [50]:
admin_ampara = (
    monthly_admin_proportional[
        monthly_admin_proportional[
            "administrative_district"
        ] == "Ampara"
    ]
    .set_index("month")["dengue_cases"]
)

In [51]:
ampara_matches = np.allclose(
    ampara_reporting_check[
        "expected_admin_ampara"
    ],
    admin_ampara
)
print(
    "Ampara aggregation correct:",
    ampara_matches
)

Ampara aggregation correct: True


Create a clean date column

In [52]:
monthly_admin_proportional[
    "date"
] = (
    monthly_admin_proportional["month"]
    .dt.to_timestamp()
)

In [53]:
monthly_admin_proportional[
    [
        "date",
        "month",
        "administrative_district",
        "dengue_cases"
    ]
].head()

,date,month,administrative_district,dengue_cases
0,2010-01-01,2010-01,Ampara,116.428571
1,2010-02-01,2010-02,Ampara,124.428571
2,2010-03-01,2010-03,Ampara,72.857143
3,2010-04-01,2010-04,Ampara,37.285714
4,2010-05-01,2010-05,Ampara,13.714286


Add calendar year and month number

In [54]:
monthly_admin_proportional["year"] = (
    monthly_admin_proportional["date"].dt.year
)
monthly_admin_proportional[
    "month_number"
] = (
    monthly_admin_proportional["date"].dt.month
)

In [55]:
monthly_admin_proportional[
    [
        "date",
        "year",
        "month_number",
        "administrative_district",
        "dengue_cases"
    ]
].head()

,date,year,month_number,administrative_district,dengue_cases
0,2010-01-01,2010,1,Ampara,116.428571
1,2010-02-01,2010,2,Ampara,124.428571
2,2010-03-01,2010,3,Ampara,72.857143
3,2010-04-01,2010,4,Ampara,37.285714
4,2010-05-01,2010,5,Ampara,13.714286


Validate final shape

In [56]:
monthly_admin_final = (
    monthly_admin_proportional[
        [
            "date",
            "year",
            "month_number",
            "administrative_district",
            "dengue_cases"
        ]
    ]
    .sort_values(
        [
            "administrative_district",
            "date"
        ]
    )
    .reset_index(drop=True)
)

monthly_admin_final.head()

,date,year,month_number,administrative_district,dengue_cases
0,2010-01-01,2010,1,Ampara,116.428571
1,2010-02-01,2010,2,Ampara,124.428571
2,2010-03-01,2010,3,Ampara,72.857143
3,2010-04-01,2010,4,Ampara,37.285714
4,2010-05-01,2010,5,Ampara,13.714286


In [57]:
print(
    "Final monthly dengue shape:",
    monthly_admin_final.shape
)

print(
    "Districts:",
    monthly_admin_final[
        "administrative_district"
    ].nunique()
)

print(
    "Months:",
    monthly_admin_final[
        "date"
    ].nunique()
)

Final monthly dengue shape: (4500, 5)
Districts: 25
Months: 180


In [58]:
monthly_validation = pd.DataFrame({
    "Validation Check": [
        "Original weekly case total",
        "Proportional monthly case total",
        "Administrative monthly case total",
        "Administrative districts",
        "Months",
        "District month observations",
        "Minimum months per district",
        "Maximum months per district",
        "Duplicate district-month keys",
        "Missing dengue values",
        "Negative dengue values",
        "Ampara aggregation correct"
    ],

    "Result": [
        original_total_cases,
        proportional_total_cases,
        admin_total_cases,
        monthly_admin_final[
            "administrative_district"
        ].nunique(),
        monthly_admin_final[
            "date"
        ].nunique(),
        len(monthly_admin_final),
        admin_months_per_district.min(),
        admin_months_per_district.max(),
        duplicate_admin_months,
        monthly_admin_final[
            "dengue_cases"
        ].isna().sum(),
        (
            monthly_admin_final[
                "dengue_cases"
            ] < 0
        ).sum(),
        ampara_matches
    ]
})

monthly_validation

,Validation Check,Result
0,Original weekly case total,714747
1,Proportional monthly case total,714747.0
2,Administrative monthly case total,714747.0
3,Administrative districts,25
4,Months,180
5,District month observations,4500
6,Minimum months per district,180
7,Maximum months per district,180
8,Duplicate district-month keys,0
9,Missing dengue values,0


In [59]:
monthly_admin_enddate = (
    dengue_enddate
    .groupby(
        [
            "administrative_district",
            "month"
        ],
        as_index=False
    )["cases"]
    .sum()
    .rename(
        columns={
            "cases":
            "dengue_cases"
        }
    )
)
monthly_admin_enddate["date"] = (
    monthly_admin_enddate[
        "month"
    ].dt.to_timestamp()
)
monthly_admin_enddate["year"] = (
    monthly_admin_enddate[
        "date"
    ].dt.year
)
monthly_admin_enddate[
    "month_number"
] = (
    monthly_admin_enddate[
        "date"
    ].dt.month
)
monthly_admin_enddate = (
    monthly_admin_enddate[
        [
            "date",
            "year",
            "month_number",
            "administrative_district",
            "dengue_cases"
        ]
    ]
    .sort_values(
        [
            "administrative_district",
            "date"
        ]
    )
    .reset_index(drop=True)
)

In [60]:
proportional_output = (
    INTERIM_DIR /
    "dengue_monthly_proportional_2010_2024.csv"
)
baseline_output = (
    INTERIM_DIR /
    "dengue_monthly_enddate_baseline_2010_2024.csv"
)

In [61]:
monthly_admin_final.to_csv(
    proportional_output,
    index=False
)
monthly_admin_enddate.to_csv(
    baseline_output,
    index=False
)
print(
    "Proportional file saved:",
    proportional_output.exists()
)
print(
    "Baseline file saved:",
    baseline_output.exists()
)

Proportional file saved: True
Baseline file saved: True


## Key Findings

1. The validated dengue dataset contains 20,306 weekly reporting-area observations and 714,747 reported dengue cases.

2. A total of 4,030 weekly reporting area observations (19.85%) cross calendar month boundaries.

3. A total of 312 weekly reporting area observations cross calendar year boundaries.

4. The end date baseline conversion and day weighted proportional conversion both preserve the original dengue case total.

5. The proportional method produces complete monthly coverage from January 2010 through December 2024 for all 26 health reporting areas.

6. The resulting reporting area dataset contains 4,680 observations, corresponding to 26 reporting areas × 180 calendar months.

7. A total of 4,269 monthly reporting area observations differ between the end-date and proportional conversion methods, demonstrating that temporal conversion methodology materially affects the constructed monthly dengue series.

8. The proportional reporting area observations were subsequently aggregated to 25 administrative districts for compatibility with district-level population data.

9. Ampara and Kalmunai health reporting observations contribute to the Ampara administrative district dengue total.

10. The administrative district dataset contains 4,500 observations, corresponding to 25 districts × 180 months.

11. Total dengue cases are preserved through both temporal conversion and administrative district aggregation.

12. Fractional monthly values generated by proportional allocation are retained without rounding to avoid introducing additional transformation error.

## Conclusion

Weekly dengue observations were successfully transformed into complete monthly time series for the period January 2010 to December 2024.

A simple end date assignment method was retained as a baseline, while a day weighted proportional allocation method was used as the improved temporal preprocessing approach. The proportional method distributes weekly case counts according to the number of reporting days falling within each calendar month and therefore explicitly handles reporting periods that cross month boundaries.

The proportional approach assumes a uniform distribution of cases across the seven days of each weekly reporting period because daily case counts are unavailable. It should therefore be interpreted as a temporal allocation strategy rather than a reconstruction of the unknown true daily case distribution.

Both conversion methods preserve the original total of 714,747 dengue cases. The proportional method provides complete monthly coverage for all reporting areas from January 2010 through December 2024.

For compatibility with district level population data, the proportionally allocated observations were aggregated from 26 health reporting areas to 25 administrative districts. Dengue case counts are additive, allowing the Ampara and Kalmunai reporting area observations to contribute to the administrative Ampara total.

The resulting administrative district dataset contains 4,500 district month observations, representing 25 districts across 180 consecutive months. No case counts were lost during temporal conversion or geographic aggregation.

Both the baseline and proportional monthly datasets were retained so that the effect of temporal preprocessing can be evaluated later under a fixed forecasting architecture.